# Ordinal Logistic Regression of Soil Classification in Each Area in Surabaya City

#Import Package

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression # For Ordinal Logistic Regression
import statsmodels.api as sm

In [ ]:
df = pd.read_excel('/content/Surabaya Barat.xlsx')

In [ ]:
df.head()

,Wilayah,Kedalaman,Konus,FR,Konsistensi
0,Barat,1,5,10.0,Soft
1,Barat,2,15,3.3,Stiff
2,Barat,3,25,4.0,Very Stiff
3,Barat,4,35,2.9,Very Stiff
4,Barat,5,40,2.5,Hard


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166 entries, 0 to 165
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Wilayah           166 non-null    object 
 1   Kedalaman         166 non-null    int64  
 2   Konus             166 non-null    int64  
 3   FR                166 non-null    float64
 4   Konsistensi       166 non-null    object 
 5   Konsistensi_Kode  166 non-null    int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 7.9+ KB


In [ ]:
df.corr(numeric_only=True)

,Kedalaman,Konus,FR
Kedalaman,1.000000,0.532061,-0.318801
Konus,0.532061,1.000000,-0.504421
FR,-0.318801,-0.504421,1.000000


In [ ]:
df.describe()
df['Konsistensi_Kode'].value_counts()

,count
Konsistensi_Kode,
5,60
2,34
4,32
1,20
3,17
0,3


In [ ]:
# Manual encoding of 'Konsistensi' column
encoding_map = {
    'Very Soft': 0,
    'Soft': 1,
    'Medium': 2,
    'Stiff': 3,
    'Very Stiff': 4,
    'Hard': 5
}

df['Konsistensi_Kode'] = df['Konsistensi'].map(encoding_map)
display(df.head())

,Wilayah,Kedalaman,Konus,FR,Konsistensi,Konsistensi_Kode
0,Barat,1,5,10.0,Soft,1
1,Barat,2,15,3.3,Stiff,3
2,Barat,3,25,4.0,Very Stiff,4
3,Barat,4,35,2.9,Very Stiff,4
4,Barat,5,40,2.5,Hard,5


Data is already prepared for used

# Soil Classification of WEST SURABAYA

In [ ]:
X = df[['Kedalaman', 'FR', 'Konus']]
y = df['Konsistensi_Kode']

# Initialize Multinomial Logistic Regression model
# 'multi_class='multinomial' indicates multinomial logistic regression
# 'solver='newton-cg' is a good choice for multinomial logistic regression
model = LogisticRegression(multi_class='multinomial', solver='newton-cg', max_iter=1000)

# Fit the model
model.fit(X, y)

# Display the coefficients and intercept
print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)

Intercept: [ 26.53919003  24.21238859  18.47136023   3.59361947 -16.11126873
 -56.70528958]
Coefficients: [[-0.36782089  0.00520833 -2.50762102]
 [-0.02187593  0.22957539 -1.87334393]
 [ 0.03235495 -0.11151033 -0.5935319 ]
 [ 0.08759874 -0.07240865  0.57035665]
 [ 0.15311422 -0.01827371  1.65188105]
 [ 0.11662891 -0.03259102  2.75225915]]


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [ ]:
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel

model = OrderedModel(y, X, distr='logit')
res = model.fit()
print(res.summary())


                             OrderedModel Results                             
Dep. Variable:       Konsistensi_Kode   Log-Likelihood:                -46.075
Model:                   OrderedModel   AIC:                             108.2
Method:            Maximum Likelihood   BIC:                             133.0
Date:                Thu, 11 Dec 2025                                         
Time:                        11:29:05                                         
No. Observations:                 166                                         
Df Residuals:                     158                                         
Df Model:                           3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Kedalaman     -0.4765      0.098     -4.870      0.000      -0.668      -0.285
FR            -0.3076      0.125     -2.465      0.0

/usr/local/lib/python3.12/dist-packages/statsmodels/base/optimizer.py:748: RuntimeWarning: Maximum number of iterations has been exceeded.
  retvals = optimize.fmin(f, start_params, args=fargs, xtol=xtol,
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [ ]:
import pandas as pd

# Ambil tabel koefisien dari summary
coef_table = res.summary().tables[1]

# Convert summary table (string format) ke DataFrame
df = pd.DataFrame(coef_table.data)
df.columns = df.iloc[0]      # Gunakan baris pertama sebagai header
df = df[1:]                  # Buang baris header dari data

# Simpan ke Excel
df.to_excel("ordinal_logit_output.xlsx", index=False)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X)

print("Accuracy:", accuracy_score(y, y_pred))

print("\nClassification Report:")
print(classification_report(y, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y, y_pred))


ValueError: shapes (166,3) and (161,3) not aligned: 3 (dim 1) != 161 (dim 0)

In [ ]:
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel

df["Konsistensi_Kode"] = pd.Categorical(df["Konsistensi_Kode"], ordered=True)
# Define the independent and dependent variables
X_ordinal = df[['Kedalaman', 'FR', 'Konus']]
y_ordinal = df['Konsistensi_Kode']
# Ensure target is treated as ordered categorical


# Initialize and fit the Ordered Logistic Regression model
# The dependent variable y_ordinal should be treated as ordered categorical
model_ordinal = OrderedModel(y_ordinal, X_ordinal, distr='logit')
model_ordinal_fit = model_ordinal.fit(disp=False)

# Print the model summary
print(model_ordinal_fit.summary())


                             OrderedModel Results                             
Dep. Variable:       Konsistensi_Kode   Log-Likelihood:                -46.075
Model:                   OrderedModel   AIC:                             108.2
Method:            Maximum Likelihood   BIC:                             133.0
Date:                Thu, 11 Dec 2025                                         
Time:                        10:34:43                                         
No. Observations:                 166                                         
Df Residuals:                     158                                         
Df Model:                           3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Kedalaman     -0.4765      0.098     -4.870      0.000      -0.668      -0.285
FR            -0.3076      0.125     -2.465      0.0

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [ ]:
import pandas as pd

# Ambil tabel koefisien dari summary
coef_table = model_sm_fit.summary().tables[1]

# Convert summary table (string format) ke DataFrame
df = pd.DataFrame(coef_table.data)
df.columns = df.iloc[0]      # Gunakan baris pertama sebagai header
df = df[1:]                  # Buang baris header dari data

# Simpan ke Excel
df.to_excel("ordinal_logit_output.xlsx", index=False)

#Soil Classification of East Surabaya

In [ ]:
df1 = pd.read_excel('/content/Surabaya Timur.xlsx')

In [ ]:
df1.head()

,Wilayah,Kedalaman,Konus,FR,Konsistensi
0,Timur,1,5,6.0,Soft
1,Timur,2,6,8.3,Medium
2,Timur,3,5,6.0,Soft
3,Timur,4,5,10.0,Soft
4,Timur,5,5,6.0,Soft


In [ ]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 563 entries, 0 to 562
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Wilayah      563 non-null    object 
 1   Kedalaman    563 non-null    int64  
 2   Konus        563 non-null    int64  
 3   FR           563 non-null    float64
 4   Konsistensi  563 non-null    object 
dtypes: float64(1), int64(2), object(2)
memory usage: 22.1+ KB


In [ ]:
# Manual encoding of 'Konsistensi' column
encoding_map = {
    'Very Soft': 0,
    'Soft': 1,
    'Medium': 2,
    'Stiff': 3,
    'Very Stiff': 4,
    'Hard': 5
}

df1['Konsistensi_Kode'] = df1['Konsistensi'].map(encoding_map)
display(df1.head())

,Wilayah,Kedalaman,Konus,FR,Konsistensi,Konsistensi_Kode
0,Timur,1,5,6.0,Soft,1
1,Timur,2,6,8.3,Medium,2
2,Timur,3,5,6.0,Soft,1
3,Timur,4,5,10.0,Soft,1
4,Timur,5,5,6.0,Soft,1


In [ ]:
import statsmodels.api as sm

# Define the independent and dependent variables again for statsmodels
X_sm = df1[['Kedalaman', 'FR', 'Konus']]
y_sm = df1['Konsistensi_Kode']


import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel

model1 = OrderedModel(y_sm, X_sm, distr='logit')
res1 = model1.fit()
print(res1.summary())


                             OrderedModel Results                             
Dep. Variable:       Konsistensi_Kode   Log-Likelihood:                -122.13
Model:                   OrderedModel   AIC:                             260.3
Method:            Maximum Likelihood   BIC:                             294.9
Date:                Thu, 11 Dec 2025                                         
Time:                        11:37:44                                         
No. Observations:                 563                                         
Df Residuals:                     555                                         
Df Model:                           3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Kedalaman      0.0757      0.040      1.909      0.056      -0.002       0.153
FR            -0.9896      0.118     -8.369      0.0

/usr/local/lib/python3.12/dist-packages/statsmodels/base/optimizer.py:748: RuntimeWarning: Maximum number of iterations has been exceeded.
  retvals = optimize.fmin(f, start_params, args=fargs, xtol=xtol,
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [ ]:
import pandas as pd

# Ambil tabel koefisien dari summary
coef_table = res1.summary().tables[1]

# Convert summary table (string format) ke DataFrame
df = pd.DataFrame(coef_table.data)
df.columns = df.iloc[0]      # Gunakan baris pertama sebagai header
df = df[1:]                  # Buang baris header dari data

# Simpan ke Excel
df.to_excel("ordinal_logit_output_east.xlsx", index=False)

In [ ]:
X = df1[['Kedalaman', 'FR']]
y = df1['Konsistensi_Kode']

# Initialize Multinomial Logistic Regression model
# 'multi_class='multinomial' indicates multinomial logistic regression
# 'solver='newton-cg' is a good choice for multinomial logistic regression
model = LogisticRegression(multi_class='multinomial', solver='newton-cg', max_iter=1000)

# Fit the model
model.fit(X, y)

# Display the coefficients and intercept
print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)

Intercept: [-1.83380452  0.27436227  1.20663358 -0.02997723 -0.48371224  0.86649813]
Coefficients: [[-0.44451867  0.67383332]
 [-0.26258403  0.55915299]
 [-0.0465779   0.21725842]
 [ 0.13341968 -0.20666154]
 [ 0.23392033 -0.17707543]
 [ 0.38634059 -1.06650776]]


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


#Soil Classification of Center Surabaya

In [ ]:
df2 = pd.read_excel('/content/Surabaya Pusat.xlsx')

In [ ]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 143 entries, 0 to 142
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Wilayah      143 non-null    object 
 1   Kedalaman    143 non-null    int64  
 2   Konus        143 non-null    int64  
 3   FR           143 non-null    float64
 4   Konsistensi  143 non-null    object 
dtypes: float64(1), int64(2), object(2)
memory usage: 5.7+ KB


In [ ]:
# Manual encoding of 'Konsistensi' column
encoding_map = {
    'Very Soft': 0,
    'Soft': 1,
    'Medium': 2,
    'Stiff': 3,
    'Very Stiff': 4,
    'Hard': 5
}

df2['Konsistensi_Kode'] = df2['Konsistensi'].map(encoding_map)
display(df2.head())

,Wilayah,Kedalaman,Konus,FR,Konsistensi,Konsistensi_Kode
0,Pusat,1,10,5.0,Medium,2
1,Pusat,2,25,2.0,Very Stiff,4
2,Pusat,3,15,3.3,Stiff,3
3,Pusat,4,10,5.0,Medium,2
4,Pusat,5,5,4.0,Soft,1


In [ ]:
import statsmodels.api as sm

# Define the independent and dependent variables again for statsmodels
X_sm = df2[['Kedalaman', 'FR', 'Konus']]
y_sm = df2['Konsistensi_Kode']


import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel

model2 = OrderedModel(y_sm, X_sm, distr='logit')
res2 = model2.fit()
print(res2.summary())


                             OrderedModel Results                             
Dep. Variable:       Konsistensi_Kode   Log-Likelihood:                -34.231
Model:                   OrderedModel   AIC:                             82.46
Method:            Maximum Likelihood   BIC:                             103.2
Date:                Thu, 11 Dec 2025                                         
Time:                        11:40:33                                         
No. Observations:                 143                                         
Df Residuals:                     136                                         
Df Model:                           3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Kedalaman     -0.4685      0.149     -3.144      0.002      -0.761      -0.176
FR            -0.2885      0.148     -1.956      0.0

/usr/local/lib/python3.12/dist-packages/statsmodels/base/optimizer.py:748: RuntimeWarning: Maximum number of iterations has been exceeded.
  retvals = optimize.fmin(f, start_params, args=fargs, xtol=xtol,
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [ ]:
import pandas as pd

# Ambil tabel koefisien dari summary
coef_table = res2.summary().tables[1]

# Convert summary table (string format) ke DataFrame
df = pd.DataFrame(coef_table.data)
df.columns = df.iloc[0]      # Gunakan baris pertama sebagai header
df = df[1:]                  # Buang baris header dari data

# Simpan ke Excel
df.to_excel("ordinal_logit_output_center.xlsx", index=False)

#Soil Classification of South Surabaya

In [ ]:
df3 = pd.read_excel('/content/Surabaya Selatan.xlsx')

In [ ]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1140 entries, 0 to 1139
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Wilayah      1140 non-null   object 
 1   Kedalaman    1140 non-null   int64  
 2   Konus        1140 non-null   float64
 3   FR           1139 non-null   float64
 4   Konsistensi  1140 non-null   object 
dtypes: float64(2), int64(1), object(2)
memory usage: 44.7+ KB


In [ ]:
# Fill missing values in 'FR' with the mean of 'FR'
mean_fr = df3['FR'].mean()
df3['FR'].fillna(mean_fr, inplace=True)

# Verify that there are no more missing values in 'FR'
print("Missing values in 'FR' after filling:")
print(df3['FR'].isnull().sum())

Missing values in 'FR' after filling:
0


/tmp/ipython-input-2508639767.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df3['FR'].fillna(mean_fr, inplace=True)


In [ ]:
# Manual encoding of 'Konsistensi' column
encoding_map = {
    'Very Soft': 0,
    'Soft': 1,
    'Medium': 2,
    'Stiff': 3,
    'Very Stiff': 4,
    'Hard': 5
}

df3['Konsistensi_Kode'] = df3['Konsistensi'].map(encoding_map)
display(df3.head())

,Wilayah,Kedalaman,Konus,FR,Konsistensi,Konsistensi_Kode
0,Selatan,1,9.0,4.4,Medium,2
1,Selatan,2,13.0,3.1,Stiff,3
2,Selatan,3,16.0,3.1,Stiff,3
3,Selatan,4,23.0,3.0,Very Stiff,4
4,Selatan,5,27.0,3.3,Very Stiff,4


In [ ]:
import statsmodels.api as sm

# Define the independent and dependent variables again for statsmodels
X_sm = df3[['Kedalaman', 'FR', 'Konus']]
y_sm = df3['Konsistensi_Kode']


import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel

model3 = OrderedModel(y_sm, X_sm, distr='logit')
res3 = model3.fit()
print(res3.summary())


                             OrderedModel Results                             
Dep. Variable:       Konsistensi_Kode   Log-Likelihood:                -896.26
Model:                   OrderedModel   AIC:                             1809.
Method:            Maximum Likelihood   BIC:                             1849.
Date:                Thu, 11 Dec 2025                                         
Time:                        11:45:18                                         
No. Observations:                1140                                         
Df Residuals:                    1132                                         
Df Model:                           3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Kedalaman      0.1628      0.012     14.114      0.000       0.140       0.185
FR            -0.2215      0.033     -6.795      0.0

/usr/local/lib/python3.12/dist-packages/statsmodels/base/optimizer.py:748: RuntimeWarning: Maximum number of iterations has been exceeded.
  retvals = optimize.fmin(f, start_params, args=fargs, xtol=xtol,
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [ ]:
import pandas as pd

# Ambil tabel koefisien dari summary
coef_table = res3.summary().tables[1]

# Convert summary table (string format) ke DataFrame
df = pd.DataFrame(coef_table.data)
df.columns = df.iloc[0]      # Gunakan baris pertama sebagai header
df = df[1:]                  # Buang baris header dari data

# Simpan ke Excel
df.to_excel("ordinal_logit_output_south.xlsx", index=False)

#Soil Classification of NORTH SURABAYA

In [ ]:
df4 = pd.read_excel('/content/Surabaya Utara.xlsx')

In [ ]:
df4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 410 entries, 0 to 409
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Wilayah      410 non-null    object 
 1   Kedalaman    410 non-null    int64  
 2   Konus        410 non-null    int64  
 3   FR           410 non-null    float64
 4   Konsistensi  410 non-null    object 
dtypes: float64(1), int64(2), object(2)
memory usage: 16.1+ KB


In [ ]:
# Manual encoding of 'Konsistensi' column
encoding_map = {
    'Very Soft': 0,
    'Soft': 1,
    'Medium': 2,
    'Stiff': 3,
    'Very Stiff': 4,
    'Hard': 5
}

df4['Konsistensi_Kode'] = df4['Konsistensi'].map(encoding_map)
display(df4.head())

,Wilayah,Kedalaman,Konus,FR,Konsistensi,Konsistensi_Kode
0,Utara,2,5,10.0,Soft,1
1,Utara,3,35,2.9,Very Stiff,4
2,Utara,4,10,5.0,Medium,2
3,Utara,5,10,5.0,Medium,2
4,Utara,6,5,6.0,Soft,1


In [ ]:
import statsmodels.api as sm

# Define the independent and dependent variables again for statsmodels
X_sm = df4[['Kedalaman', 'FR', 'Konus']]
y_sm = df4['Konsistensi_Kode']


import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel

model4 = OrderedModel(y_sm, X_sm, distr='logit')
res4 = model4.fit()
print(res4.summary())


                             OrderedModel Results                             
Dep. Variable:       Konsistensi_Kode   Log-Likelihood:                -192.15
Model:                   OrderedModel   AIC:                             400.3
Method:            Maximum Likelihood   BIC:                             432.4
Date:                Thu, 11 Dec 2025                                         
Time:                        11:44:32                                         
No. Observations:                 410                                         
Df Residuals:                     402                                         
Df Model:                           3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Kedalaman     -0.2505      0.035     -7.087      0.000      -0.320      -0.181
FR            -0.3556      0.068     -5.262      0.0

/usr/local/lib/python3.12/dist-packages/statsmodels/base/optimizer.py:748: RuntimeWarning: Maximum number of iterations has been exceeded.
  retvals = optimize.fmin(f, start_params, args=fargs, xtol=xtol,
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [ ]:
import pandas as pd

# Ambil tabel koefisien dari summary
coef_table = res4.summary().tables[1]

# Convert summary table (string format) ke DataFrame
df = pd.DataFrame(coef_table.data)
df.columns = df.iloc[0]      # Gunakan baris pertama sebagai header
df = df[1:]                  # Buang baris header dari data

# Simpan ke Excel
df.to_excel("ordinal_logit_output_north.xlsx", index=False)